In [1]:
import sys
sys.path.append("..")

import time
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

PROC_DIR = Path("../data/processed")
df = pd.read_csv(PROC_DIR / "tickets_clean_en.csv")

# exact same split as the TF-IDF models — non-negotiable for a fair comparison
tr, te = train_test_split(df, test_size=0.2, random_state=42,
                          stratify=df["queue"])
print("Train:", len(tr), "| Test:", len(te))

Train: 9520 | Test: 2381


In [2]:
from sentence_transformers import SentenceTransformer

# multilingual model: also lets us say something concrete about French support
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

t0 = time.perf_counter()
Etr = model.encode(tr["text"].tolist(), batch_size=64,
                   show_progress_bar=True, convert_to_numpy=True)
Ete = model.encode(te["text"].tolist(), batch_size=64,
                   show_progress_bar=True, convert_to_numpy=True)
encode_time = time.perf_counter() - t0

print(f"\nEmbedding shape: {Etr.shape}")
print(f"Encoding time: {encode_time:.1f}s for {len(tr)+len(te)} tickets")
print(f"Per ticket: {1000*encode_time/(len(tr)+len(te)):.2f} ms")

c:\Users\Lenovo LOQ\Desktop\PFA_2026\intelligent-helpdesk-api\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lenovo LOQ\Desktop\PFA_2026\intelligent-helpdesk-api\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo LOQ\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows,


Embedding shape: (9520, 384)
Encoding time: 111.2s for 11901 tickets
Per ticket: 9.34 ms


In [3]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

rows = []
for C in [0.5, 1.0, 2.0, 4.0, 8.0]:
    m = LinearSVC(class_weight="balanced", C=C, max_iter=10000).fit(Etr, tr["queue"])
    p = m.predict(Ete)
    rows.append({"C": C,
                 "acc": accuracy_score(te["queue"], p),
                 "macroF1": f1_score(te["queue"], p, average="macro")})

sweep = pd.DataFrame(rows)
print(sweep.round(4).to_string(index=False))
print("\nTF-IDF reference (C=4.0): acc=0.5489  macroF1=0.5336")

  C    acc  macroF1
0.5 0.3230   0.2834
1.0 0.3293   0.2895
2.0 0.3314   0.2915
4.0 0.3314   0.2928
8.0 0.3326   0.2919

TF-IDF reference (C=4.0): acc=0.5489  macroF1=0.5336


In [4]:
rows = []
for C in [0.5, 1.0, 2.0, 4.0, 8.0]:
    m = LinearSVC(class_weight="balanced", C=C, max_iter=10000).fit(Etr, tr["priority"])
    p = m.predict(Ete)
    rows.append({"C": C,
                 "acc": accuracy_score(te["priority"], p),
                 "macroF1": f1_score(te["priority"], p, average="macro")})

print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nTF-IDF reference (C=4.0): acc=0.6144  macroF1=0.5973")

  C    acc  macroF1
0.5 0.4813   0.4599
1.0 0.4780   0.4573
2.0 0.4717   0.4518
4.0 0.4704   0.4500
8.0 0.4708   0.4506

TF-IDF reference (C=4.0): acc=0.6144  macroF1=0.5973


In [5]:
gold = pd.read_csv(PROC_DIR / "gold_analysis.csv")
Eg = model.encode(gold["text"].tolist(), batch_size=64, convert_to_numpy=True)

BEST_C = 2.0  # replace with your sweep winner
best_q = LinearSVC(class_weight="balanced", C=BEST_C, max_iter=10000).fit(Etr, tr["queue"])
pred_g = best_q.predict(Eg)

print(f"Embeddings vs corpus labels : {accuracy_score(gold['data_queue'], pred_g):.4f}")
print(f"Embeddings vs human labels  : {accuracy_score(gold['human_queue'], pred_g):.4f}")
print(f"\nTF-IDF vs corpus : 0.4933")
print(f"TF-IDF vs human  : 0.2667")

Embeddings vs corpus labels : 0.4000
Embeddings vs human labels  : 0.2933

TF-IDF vs corpus : 0.4933
TF-IDF vs human  : 0.2667


In [6]:
raw = pd.read_csv("../data/raw/dataset-tickets-multi-lang3-4k.csv")
fr = raw[raw["language"] == "fr"].copy()

from src.preprocessing import build_ticket_text
fr["text"] = [build_ticket_text(s, b) for s, b in zip(fr["subject"], fr["body"])]
fr = fr[fr["text"].str.split().str.len() >= 5]

Efr = model.encode(fr["text"].tolist(), batch_size=64, convert_to_numpy=True)
pred_fr = best_q.predict(Efr)

print(f"French tickets: {len(fr)}")
print(f"Accuracy of an English-trained model on French text: "
      f"{accuracy_score(fr['queue'], pred_fr):.4f}")

French tickets: 476
Accuracy of an English-trained model on French text: 0.1660


In [7]:
np.save(PROC_DIR / "emb_train.npy", Etr)
np.save(PROC_DIR / "emb_test.npy", Ete)
print("Embeddings cached")

Embeddings cached


In [8]:
raw_fr = raw[raw["language"] == "fr"].copy()
raw_fr["raw_text"] = (raw_fr["subject"].fillna("") + " " + raw_fr["body"].fillna("")).str.strip()
raw_fr = raw_fr[raw_fr["raw_text"].str.split().str.len() >= 5]

Efr = model.encode(raw_fr["raw_text"].tolist(), batch_size=64, convert_to_numpy=True)

# retrain on RAW English text for a like-for-like comparison
raw_en = pd.read_csv("../data/raw/dataset-tickets-multi-lang-4-20k.csv")
raw_en = raw_en[raw_en["language"] == "en"].copy()
raw_en["raw_text"] = (raw_en["subject"].fillna("") + " " + raw_en["body"].fillna("")).str.strip()
raw_en = raw_en[raw_en["raw_text"].str.split().str.len() >= 5]

Een = model.encode(raw_en["raw_text"].tolist(), batch_size=64,
                   show_progress_bar=True, convert_to_numpy=True)
m_raw = LinearSVC(class_weight="balanced", C=4.0, max_iter=10000).fit(Een, raw_en["queue"])

print(f"EN (in-sample):  {accuracy_score(raw_en['queue'], m_raw.predict(Een)):.4f}")
print(f"FR (zero-shot):  {accuracy_score(raw_fr['queue'], m_raw.predict(Efr)):.4f}")

Batches: 100%|██████████| 187/187 [02:11<00:00,  1.42it/s]


EN (in-sample):  0.4094
FR (zero-shot):  0.1954
